# Exploratory Data Analysis & Setup
This cell will install libraries, generate audio files, and create `metadata.csv` automatically in Colab.


In [ ]:
!pip install librosa numpy pandas matplotlib scikit-learn tensorflow seaborn

import os
import librosa
import librosa.display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.io.wavfile as wavfile
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')

# 1. GENERATE DATASET WITH CSV METADATA
def create_dataset():
    classes = {'crying': 1.5, 'hungry': 1.0, 'discomfort': 0.8, 'sleeping': 0.2}
    base_dir = "data/raw"
    os.makedirs('data', exist_ok=True)
    os.makedirs(base_dir, exist_ok=True)
    metadata = []
    
    for cls, freq_mod in classes.items():
        cls_dir = os.path.join(base_dir, cls)
        os.makedirs(cls_dir, exist_ok=True)
        for i in range(25):
            t = np.linspace(0, 2.0, int(22050 * 2.0))
            signal = np.sin(2 * np.pi * 440 * freq_mod * t) + np.random.normal(0, 0.5, len(t))
            signal = signal / np.max(np.abs(signal))
            signal += np.random.normal(0, 0.1, len(signal))
            signal = np.int16(signal * 32767)
            
            filepath = os.path.join(cls_dir, f"sample_{i:03d}.wav")
            wavfile.write(filepath, 22050, signal)
            metadata.append({'filename': filepath, 'class': cls})
            
    pd.DataFrame(metadata).to_csv('data/metadata.csv', index=False)
    print("Dataset generation complete. Mappings successfully saved to 'data/metadata.csv'.")

if not os.path.exists('data/metadata.csv'):
    create_dataset()

# 2. FEATURE EXTRACTION FUNCTIONS
def extract_ml_features(audio, sr=22050):
    mfcc = np.mean(librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40).T, axis=0)
    zcr = np.mean(librosa.feature.zero_crossing_rate(y=audio).T, axis=0)
    chroma = np.mean(librosa.feature.chroma_stft(S=np.abs(librosa.stft(audio)), sr=sr).T, axis=0)
    return np.hstack([mfcc, zcr, chroma])

def extract_dl_features(audio, sr=22050, max_pad_len=100):
    mel_spec_db = librosa.power_to_db(librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128), ref=np.max)
    if mel_spec_db.shape[1] > max_pad_len:
        return mel_spec_db[:, :max_pad_len]
    else:
        pad_width = max_pad_len - mel_spec_db.shape[1]
        return np.pad(mel_spec_db, pad_width=((0,0), (0, pad_width)), mode='constant')

def load_dataset(feature_type='ml'):
    df = pd.read_csv('data/metadata.csv')
    classes = sorted(df['class'].unique())
    class_map = {label: idx for idx, label in enumerate(classes)}
    data, labels = [], []
    
    for _, row in df.iterrows():
        audio, sr = librosa.load(row['filename'], sr=22050)
        # Add original and augmented
        for a in [audio, audio + 0.005 * np.random.randn(len(audio))]:
            feats = extract_ml_features(a, sr) if feature_type == 'ml' else extract_dl_features(a, sr)
            data.append(feats)
            labels.append(class_map[row['class']])
            
    return np.array(data), np.array(labels), class_map


### View Metadata CSV


In [ ]:
df = pd.read_csv('data/metadata.csv')
df.head()


### Visualize Waveforms and Spectrograms


In [ ]:
classes = df['class'].unique()
for c in classes:
    sample_file = df[df['class'] == c]['filename'].iloc[0]
    y, sr = librosa.load(sample_file, sr=22050)
    
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    librosa.display.waveshow(y, sr=sr)
    plt.title(f'Waveform: {c}')
    
    plt.subplot(1, 2, 2)
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='log')
    plt.colorbar(format='%+2.0f dB')
    plt.title(f'Spectrogram: {c}')
    plt.tight_layout()
    plt.show()
